In [ ]:
# Importing necessary libraries and py files
import os # Utilities
import glob # File discovery
import torch # PyTorch machine learning
import numpy as np # Numerical operations
import pandas as pd # Data manipulation

# Importing from created .py files
import data_loader # Loads data

In [ ]:
# Defining function to convert folder of per-variant .pt files into compatible pkl file for pipeline
def convertPtFolder(dataset, model_name, pt_dir, verbose=True):
    save_path = f"{dataset}_{model_name}_embeddings.pkl" # Setting save file name

    # Checking if dataset already converted
    if os.path.exists(save_path):
        print(f"Embedding dataset already exists at {save_path}, loading instead") # Signalling skip
        full_df = pd.read_pickle(save_path) # Loading dataframe
        print("Embeddings loaded\n") # Signalling function end

        return full_df

    # Converting per-variant files and creating dataset if not
    else:
        print("Converting embedding dataset...") # Signalling conversion start

        files = sorted(glob.glob(os.path.join(pt_dir, "*.pt"))) # Getting per-variant files
        if len(files) == 0:
            raise FileNotFoundError(f"No .pt files found in {pt_dir}") # Guarding against empty input

        embeddings = {} # Initialising dictionary to store embeddings

        # Iterating through per-variant files
        for f in files:
            obj = torch.load(f, map_location="cpu", weights_only=False) # Loading one variant
            label = obj["label"] # Variant label

            # Taking the mean representation from final layers
            (layer, tensor), = obj["mean_representations"].items()

            # Saving embeddings as plain numpy array to match other module outputs
            embeddings[label] = tensor.float().numpy()

        # Building dataset with variant label, ESM embeddings, and score (NA if missing)
        rows = [] # Initialising rows
        for label, embedding in embeddings.items():
            row = {"variant": label} # Initialising rows with variant label
            for i in range(len(embedding)):
                row[f"feature_{i}"] = embedding[i] # Adding feature values
            rows.append(row) # Saving full row

        full_df = pd.DataFrame(rows) # Building dataframe

        wild_type, scores_df = data_loader.loadDataset(dataset, verbose) # Loading scores data
        full_df = full_df.merge(scores_df[["variant", "score"]], on="variant", how="left") # Merging in scores

        full_df.to_pickle(save_path) # Saving embeddings so they don't need to be reconverted

        print(f"Embeddings converted, loaded, and saved to {save_path}") # Signalling function end

        return full_df

In [ ]:
# Example usage:
# Converting ESM-2 15B embeddings for P53
full_df = convertPtFolder(
    dataset="P53", # Dataset name
    model_name="esm2_t48_15B_UR50D", # Encoder calling string
    verbose=True,
)

full_df.head() # Previewing first few rows